# Task 3 · Cleaning Data
**Track:** Data Analytics | **Level:** 1

**Objective:** Demonstrate professional-level data cleaning skills by taking a deliberately
messy dataset and systematically transforming it into a clean, analysis-ready dataset.
Every decision is documented below.

**Dataset:** `messy_employee_data.csv` — a synthetic, deliberately-dirtied employee/HR
dataset (inconsistent casing, mixed date formats, duplicates, nulls, outliers, wrong
dtypes) — in the spirit of the "messy data cleaning exercise" style datasets on Kaggle.

**Tech Stack:** Python, pandas, numpy


In [1]:

import numpy as np
import pandas as pd

np.random.seed(11)

# ------------------------------------------------------------------
# Build a deliberately messy dataset (generated locally; no internet
# access in this environment to pull a Kaggle CSV directly).
# ------------------------------------------------------------------
n = 500
names = [f"Employee_{i}" for i in range(1, n + 1)]
depts = ["Sales", "sales", "SALES", "Engineering", "engineering", "HR", "Hr", "Marketing"]
genders = ["Male", "male", "M", "Female", "female", "F"]

date_formats = lambda d: np.random.choice([
    d.strftime("%Y-%m-%d"), d.strftime("%d/%m/%Y"), d.strftime("%m-%d-%Y"), d.strftime("%d-%b-%Y")
])

rows = []
for i in range(n):
    emp_id = i + 1
    name = names[i]
    dept = np.random.choice(depts)
    gender = np.random.choice(genders)
    join_date_dt = pd.Timestamp("2018-01-01") + pd.Timedelta(days=int(np.random.uniform(0, 2500)))
    join_date = date_formats(join_date_dt)
    age = np.random.normal(34, 8)
    if np.random.rand() < 0.02:
        age = np.random.choice([150, -5, 999])  # clear outlier/garbage
    salary = np.random.normal(65000, 18000)
    if np.random.rand() < 0.015:
        salary = salary * 20  # outlier
    salary_str = f"${salary:,.2f}" if np.random.rand() < 0.4 else round(salary, 2)
    rows.append([emp_id, name, dept, gender, join_date, round(age, 1) if not pd.isna(age) else age, salary_str])

df = pd.DataFrame(rows, columns=["EmployeeID", "Name", "Department", "Gender",
                                  "JoinDate", "Age", "Salary"])

# inject nulls
for col in ["Department", "Gender", "Age", "Salary"]:
    idx = df.sample(frac=0.05, random_state=np.random.randint(1000)).index
    df.loc[idx, col] = np.nan

# inject duplicate rows
df = pd.concat([df, df.sample(20, random_state=3)], ignore_index=True)

df.to_csv("messy_employee_data.csv", index=False)
print("Dataset saved -> messy_employee_data.csv | shape:", df.shape)
df.head()


Dataset saved -> messy_employee_data.csv | shape: (520, 7)


## 1. Load dataset and produce a 'data quality report'

In [2]:

df = pd.read_csv("messy_employee_data.csv")

def data_quality_report(data):
    report = pd.DataFrame({
        "dtype": data.dtypes,
        "nulls": data.isnull().sum(),
        "null_pct": (data.isnull().mean() * 100).round(1),
    })
    return report

print("Rows:", len(df), "| Duplicate rows:", df.duplicated().sum())
data_quality_report(df)


Rows: 520 | Duplicate rows: 20


**Value range anomalies check** (Age, Salary) before cleaning:

In [3]:

df["Age_numeric_preview"] = pd.to_numeric(df["Age"], errors="coerce")
print("Age min/max (raw):", df["Age_numeric_preview"].min(), df["Age_numeric_preview"].max())
df.drop(columns=["Age_numeric_preview"], inplace=True)


Age min/max (raw): -5.0 999.0


## 2. Missing data handling
**Decisions (documented):**
- `Department`, `Gender`: categorical -> impute with the column **mode**, since these are
  low-cardinality labels where the most frequent category is a reasonable default and
  dropping rows would lose otherwise-valid salary/age data.
- `Age`: numeric, roughly symmetric -> impute with the **median** (robust to the outliers
  we'll clip in step 4) rather than mean.
- `Salary`: numeric, right-skewed -> impute with the **median** for the same reason.


In [4]:

for col in ["Department", "Gender"]:
    mode_val = df[col].mode(dropna=True)[0]
    df[col] = df[col].fillna(mode_val)

# Salary needs numeric parsing before imputing (handled in standardisation step),
# so we impute Age now and defer Salary imputation until after type correction.
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Age"] = df["Age"].fillna(df["Age"].median())
print("Remaining nulls:\n", df.isnull().sum())


Remaining nulls:
 EmployeeID     0
Name           0
Department     0
Gender         0
JoinDate       0
Age            0
Salary        25
dtype: int64


## 3. Duplicate removal

In [5]:

before = len(df)
df = df.drop_duplicates(subset=["EmployeeID"])
after = len(df)
print(f"Removed {before - after} duplicate rows (by EmployeeID). Remaining rows: {after}")


Removed 20 duplicate rows (by EmployeeID). Remaining rows: 500


## 4. Standardisation — inconsistent formatting

In [6]:

# Department & Gender casing
df["Department"] = df["Department"].str.strip().str.title().replace({"Hr": "HR"})
df["Gender"] = df["Gender"].str.strip().str.upper().replace({
    "MALE": "Male", "M": "Male", "FEMALE": "Female", "F": "Female"
})

# JoinDate -> unified datetime (mixed formats: ISO, dd/mm/yyyy, mm-dd-yyyy, dd-Mon-yyyy)
def parse_mixed_date(s):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d-%b-%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df["JoinDate"] = df["JoinDate"].apply(parse_mixed_date)
print("Unparseable dates after standardisation:", df["JoinDate"].isnull().sum())
print(df[["Department", "Gender", "JoinDate"]].head())


Unparseable dates after standardisation: 0
  Department  Gender   JoinDate
0      Sales    Male 2022-07-20
1      Sales  Female 2023-10-28
2      Sales    Male 2024-08-22
3      Sales  Female 2018-10-07
4      Sales  Female 2022-11-30


## 5. Outlier detection (IQR method) — Age and Salary

In [7]:

# Salary: strip $ and commas, coerce to float first
df["Salary"] = (df["Salary"].astype(str).str.replace(r"[$,]", "", regex=True))
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "Salary"]:
    low, high = iqr_bounds(df[col])
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: IQR bounds = [{low:.1f}, {high:.1f}] -> {n_out} outliers detected")
    # Decision: cap (winsorize) rather than drop, to preserve sample size
    df[col] = df[col].clip(lower=max(low, 0 if col == "Age" else low), upper=high)

print("\nDecision: outliers were CAPPED (winsorized) at the IQR bounds rather than")
print("removed, since dropping rows would lose valid Department/Gender information")
print("tied to those employees.")


Age: IQR bounds = [13.9, 53.6] -> 16 outliers detected
Salary: IQR bounds = [21221.6, 111262.1] -> 10 outliers detected

Decision: outliers were CAPPED (winsorized) at the IQR bounds rather than
removed, since dropping rows would lose valid Department/Gender information
tied to those employees.


## 6. Data type correction

In [8]:

df["EmployeeID"] = df["EmployeeID"].astype(str)
df["Age"] = df["Age"].round(0).astype(int)
df["Salary"] = df["Salary"].round(2).astype(float)
df["JoinDate"] = pd.to_datetime(df["JoinDate"])
print(df.dtypes)


EmployeeID               str
Name                     str
Department               str
Gender                   str
JoinDate      datetime64[us]
Age                    int64
Salary               float64
dtype: object


## 7. Before vs. After summary table

In [9]:

raw = pd.read_csv("messy_employee_data.csv")
summary = pd.DataFrame({
    "Metric": ["Row count", "Duplicate rows", "Total null cells", "Correct dtypes (Age/Salary numeric, JoinDate datetime)"],
    "Before": [len(raw), raw.duplicated().sum(), int(raw.isnull().sum().sum()), "No"],
    "After": [len(df), df.duplicated().sum(), int(df.isnull().sum().sum()), "Yes"],
})
summary


## 8. Save the cleaned dataset

In [10]:

df.to_csv("cleaned_employee_data.csv", index=False)
print("Cleaned dataset saved -> cleaned_employee_data.csv")
df.head()


Cleaned dataset saved -> cleaned_employee_data.csv
